In [3]:
import pandas as pd
import numpy as np
import re

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix,
    classification_report
)

import pickle

In [4]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input,
    Embedding,
    Bidirectional,
    LSTM,
    Dense,
    Dropout,
    Concatenate
)

from tensorflow.keras.callbacks import EarlyStopping

In [5]:
df=pd.read_csv("shoes_reviews.csv")
df.head()

,review_text,rating,label
0,Truly comfortable hiking boots. The attention ...,5,1
1,Can't go wrong with the classic Chuck!,5,0
2,Loved the shoe - super comfortable. Only probl...,3,0
3,LOOOOVE these. My 2 year old love these as we...,5,0
4,worst worst worst. Do not buy. worst.,1,1


In [6]:
print(df.shape)
df.isnull().sum()

(1821, 3)


review_text    0
rating         0
label          0
dtype: int64

In [7]:
df["label"].value_counts()

label
0    968
1    853
Name: count, dtype: int64

In [8]:
X_text=df["review_text"]
X_rating=df["rating"]

y=df["label"]

(
    X_text_train,
    X_text_test,

    X_rating_train,
    X_rating_test,

    y_train,
    y_test
) = train_test_split(
    X_text,
    X_rating,
    y,

    test_size=0.2,
    random_state=42
)

In [10]:
tokenizer=Tokenizer(
    num_words=10000,
    oov_token="<OOV>" #Out Of Vocabulary
)

tokenizer.fit_on_texts(X_text_train)

In [11]:
X_train_seq=tokenizer.texts_to_sequences(X_text_train)
X_test_seq=tokenizer.texts_to_sequences(X_text_test)

print(X_text_train.iloc[0])
print(X_train_seq[0])

Solid pair of sneakers.
[540, 38, 14, 100]


In [12]:
review_lengths=[len(seq) for seq in X_train_seq]

print("Average length: ", np.mean(review_lengths))
print("Max length: ", np.max(review_lengths))

Average length:  31.9375
Max length:  393


In [13]:
max_length=120

X_train_pad=pad_sequences(
    X_train_seq,
    maxlen=max_length,
    padding='post',
    truncating='post'
)

X_test_pad=pad_sequences(
    X_test_seq,
    maxlen=max_length,
    padding='post',
    truncating='post'
)

print(X_train_pad.shape)
print(X_test_pad.shape)

(1456, 120)
(365, 120)


In [18]:
X_rating_train=np.array(X_rating_train)
X_rating_test=np.array(X_rating_test)

In [16]:
vocab_size=len(tokenizer.word_index)+1
print("Vocabulary size: ", vocab_size)

Vocabulary size:  3529


In [26]:
#text input---------------
text_input=Input(shape=(max_length,))

embedding=Embedding(
    input_dim=vocab_size,
    output_dim=128
)(text_input)

bilstm=Bidirectional(
    LSTM(64, dropout=0.3)
)(embedding)

text_dense=Dense(64, activation='relu')(bilstm)

#rating input-------------
rating_input=Input(shape=(1,))

rating_dense=Dense(16, activation='relu')(rating_input)

#concatenate both---------
combined=Concatenate()([
    text_dense,
    rating_dense
])

x=Dense(64, activation='relu')(combined)

x=Dropout(0.3)(x)

output=Dense(1, activation='sigmoid')(x)

In [27]:
model=Model(
    inputs=[text_input, rating_input],
    outputs=output
)

In [28]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [29]:
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer_4 (InputLayer)    │ (None, 120)               │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ embedding_2 (Embedding)       │ (None, 120, 128)          │         451,712 │ input_layer_4[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ bidirectional_2               │ (None, 128)               │          98,816 │ embedding_2[0][0]          │
│ (Bidirectional)               │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ input_layer_5 (InputLayer)    │ (None, 1)                 │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_6 (Dense)               │ (None, 64)                │           8,256 │ bidirectional_2[0][0]      │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_7 (Dense)               │ (None, 16)                │              32 │ input_layer_5[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ concatenate_2 (Concatenate)   │ (None, 80)                │               0 │ dense_6[0][0],             │
│                               │                           │                 │ dense_7[0][0]              │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_8 (Dense)               │ (None, 64)                │           5,184 │ concatenate_2[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dropout_1 (Dropout)           │ (None, 64)                │               0 │ dense_8[0][0]              │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_9 (Dense)               │ (None, 1)                 │              65 │ dropout_1[0][0]            │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 564,065 (2.15 MB)

 Trainable params: 564,065 (2.15 MB)

 Non-trainable params: 0 (0.00 B)

In [30]:
early_stop=EarlyStopping(
    monitor='val_loss',
    patience=4,
    restore_best_weights=True
)

In [32]:
history=model.fit(
    [X_train_pad, X_rating_train],
    y_train,

    validation_data=(
        [X_test_pad, X_rating_test],
        y_test
    ),

    epochs=10,
    batch_size=32,

    callbacks=[early_stop]
)

Epoch 1/10
46/46 ━━━━━━━━━━━━━━━━━━━━ 6s 51ms/step - accuracy: 0.7177 - loss: 0.5255 - val_accuracy: 0.9068 - val_loss: 0.2928
Epoch 2/10
46/46 ━━━━━━━━━━━━━━━━━━━━ 2s 41ms/step - accuracy: 0.9396 - loss: 0.1640 - val_accuracy: 0.9699 - val_loss: 0.0820
Epoch 3/10
46/46 ━━━━━━━━━━━━━━━━━━━━ 2s 41ms/step - accuracy: 0.9760 - loss: 0.0655 - val_accuracy: 0.9616 - val_loss: 0.0887
Epoch 4/10
46/46 ━━━━━━━━━━━━━━━━━━━━ 2s 40ms/step - accuracy: 0.9835 - loss: 0.0546 - val_accuracy: 0.9589 - val_loss: 0.1318
Epoch 5/10
46/46 ━━━━━━━━━━━━━━━━━━━━ 2s 40ms/step - accuracy: 0.9911 - loss: 0.0331 - val_accuracy: 0.9589 - val_loss: 0.1476
Epoch 6/10
46/46 ━━━━━━━━━━━━━━━━━━━━ 2s 40ms/step - accuracy: 0.9918 - loss: 0.0194 - val_accuracy: 0.9589 - val_loss: 0.1501


In [34]:
loss, accuracy=model.evaluate([X_test_pad, X_rating_test], y_test)

print("Test loss: ", loss)
print("Test accuracy: ", accuracy)

12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.9699 - loss: 0.0820
Test loss:  0.08199227601289749
Test accuracy:  0.9698629975318909


In [35]:
y_pred_probs=model.predict([X_test_pad, X_rating_test])
y_pred=(y_pred_probs>0.5).astype(int)

12/12 ━━━━━━━━━━━━━━━━━━━━ 1s 53ms/step


In [36]:
cm=confusion_matrix(y_test, y_pred)
print(cm)

[[182   9]
 [  2 172]]


In [37]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.99      0.95      0.97       191
           1       0.95      0.99      0.97       174

    accuracy                           0.97       365
   macro avg       0.97      0.97      0.97       365
weighted avg       0.97      0.97      0.97       365



In [38]:
model.save("fake_review_bilstm_v2.keras")

In [39]:
with open("review_tokenizer_v2.pkl", "wb") as f:
    pickle.dump(tokenizer, f)